In [1]:
import numpy as np
import os
from skimage import io
from skimage.transform import resize

In [48]:
input_landslide_images='./Bijie-landslide-dataset/landslide/image/'
input_landslide_dem='./Bijie-landslide-dataset/landslide/dem/'
input_landslide_dem_resized='./Bijie-landslide-dataset/landslide/dem_resized/'
input_landslide_mask='./Bijie-landslide-dataset/landslide/mask/'
input_landslide_images_crop='./Bijie-landslide-dataset/landslide/image_crop/'
input_landslide_dem_resized_crop='./Bijie-landslide-dataset/landslide/dem_resized_crop/'
input_landslide_mask_crop='./Bijie-landslide-dataset/landslide/mask_crop/'


input_nolandslide_images='./Bijie-landslide-dataset/non-landslide/image/'
input_nolandslide_dem='./Bijie-landslide-dataset/non-landslide/dem/'
input_nolandslide_dem_resized='./Bijie-landslide-dataset/non-landslide/dem_resized'
input_nolandslide_images_crop='./Bijie-landslide-dataset/non-landslide/image_crop/'
input_nolandslide_dem_resized_crop='./Bijie-landslide-dataset/non-landslide/dem_resized_crop'


# Images sizes and resize
### resize DEM to initial images

In [31]:
def resize_dem(input_dem,input_image,output_folder):
    """
    Resize dem images to initial images and save to output_folder
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print('creation of ',output_folder)
    names=os.listdir(input_dem)
    
    for name in names:
        s_ima=io.imread('%s/%s'%(input_image,name)).shape
        dem=io.imread('%s/%s'%(input_dem,name))
        dem_resized = resize(dem, s_ima,anti_aliasing=True)
        name_save='%s/%s'%(output_folder,name)
        data=np.clip(dem_resized*255,0,255).astype(np.uint8)
        io.imsave(name_save,data)
    print('converted %d files'%len(names))

In [30]:
resize_dem(input_landslide_dem,input_landslide_images,input_landslide_dem_resized)

creation of  ./Bijie-landslide-dataset/landslide/dem_resized/


In [32]:
resize_dem(input_nolandslide_dem,input_nolandslide_images,input_nolandslide_dem_resized)

creation of  ./Bijie-landslide-dataset/non-landslide/dem_resized


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/1180175136.py:16: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized/zb3dzj001.png is a low contrast image
  io.imsave(name_save,data)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/1180175136.py:16: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized/zb3dzj002.png is a low contrast image
  io.imsave(name_save,data)
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/1180175136.py:16: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized/zb3dzj003.png is a low contrast image
  io.imsave(name_save,data)


converted 2003 files


### check consistency with image tiles and get min,max sizes

In [33]:
def check_size_consistency1(input_images,input_dsm,input_mask):
    names=os.listdir(input_dsm)
    max_lig = 0
    max_col = 0
    min_lig=9999999
    min_col=9999999
    for name in names:
        s_ima=io.imread('%s/%s'%(input_images,name)).shape
        s_dsm=io.imread('%s/%s'%(input_dsm,name)).shape
        s_mas=io.imread('%s/%s'%(input_mask,name)).shape
        condition = ((s_ima[1]==s_dsm[1]) and (s_ima[0]==s_dsm[0])and  (s_ima[1]==s_mas[1]) and (s_ima[0]==s_mas[0]))
        if condition is False:
            print('err size for',name)
            print('size mask : ',s_mas)
            print('size dsm : ',s_dsm)
            print('size image : ',s_ima)
        if s_ima[0] > max_lig:
            max_lig=s_ima[0]
        if s_ima[1] > max_col:
            max_col=s_ima[1]
        if s_ima[0] < min_lig:
            min_lig=s_ima[0]
        if s_ima[1] < min_col:
            min_col=s_ima[1]
       
            
    print('checked ',len(names),' images')
    return min_lig,min_col,max_lig,max_col
        
def check_size_consistency2(input_images,input_dsm):
    names=os.listdir(input_dsm)
    max_lig = 0
    max_col = 0
    min_lig=9999999
    min_col=9999999
    for name in names:
        s_ima=io.imread('%s/%s'%(input_images,name)).shape
        s_dsm=io.imread('%s/%s'%(input_dsm,name)).shape
        condition = ((s_ima[1]==s_dsm[1]) and (s_ima[0]==s_dsm[0]))
        if condition is False:
            print('err size for',name)
            print('size dsm : ',s_dsm)
            print('size image : ',s_ima)
        if s_ima[0] > max_lig:
            max_lig=s_ima[0]
        if s_ima[1] > max_col:
            max_col=s_ima[1]
        if s_ima[0] < min_lig:
            min_lig=s_ima[0]
        if s_ima[1] < min_col:
            min_col=s_ima[1]

    print('checked ',len(names),' images')
    return min_lig,min_col,max_lig,max_col


In [35]:
min_lig,min_col,max_lig,max_col=check_size_consistency1(input_landslide_images,input_landslide_dem_resized,input_landslide_mask)

checked  770  images


In [37]:
min_lig,min_col,max_lig,max_col=check_size_consistency2(input_nolandslide_images,input_nolandslide_dem_resized)

checked  2003  images


### crop large images

In [38]:
def sliding_window(image, stepSize, windowSize):
  for y in range(0, image.shape[0], stepSize):
    for x in range(0, image.shape[1], stepSize):
      yield (x, y, image[y:y + windowSize[1], x:x + windowSize[0]])

./Bijie-landslide-dataset/landslide/dem_resized//ny050.png


In [44]:
def crop_image(name_im,dest_name,windowsize,overlap):
    """
    crop an image (png or jpg) with overlap and a given sliding window
    save results in dest_name

    """
    if not os.path.exists(dest_name):
        os.makedirs(dest_name)
        print('creation of ',dest_name)
    im=io.imread(name_im)
    windows=sliding_window(im, windowsize-overlap, (windowsize,windowsize))
    
    # generic image name and extension
    name=os.path.splitext(os.path.split(name_im)[1])[0]
    ext=os.path.splitext(os.path.split(name_im)[1])[1]
    num_im=0
    for window in windows:
        name_save='%s/%s_%.4d%s'%(dest_name,name,num_im,ext)
        io.imsave(name_save,window[2])
        num_im+=1


test

In [46]:
name_im='%s/ny050.png'%input_landslide_dem_resized
os.path.splitext(os.path.split(name_im)[1])[1]
print(name_im)
crop_image(name_im,'/Users/corpetti/tmpo/zfdsdf_dsm/',256,0)

./Bijie-landslide-dataset/landslide/dem_resized//ny050.png


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0004.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0012.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0016.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0017.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: /Users/corpetti/tmpo/zfdsdf_dsm//ny050_0018.png is a low contrast image
  io.imsave(name_save,window

### crop all small images in a given folder

In [54]:
def crop_images(input_folder,output_folder,window_size,overlap):
    """
    put all images in a given folder
    """
    names=os.listdir(input_folder)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print('creation of ',output_folder)

    for name in names:
        s_ima=io.imread('%s/%s'%(input_folder,name)).shape
        if (s_ima[0]> window_size) or (s_ima[1]> window_size):
            crop_image('%s/%s'%(input_folder,name),output_folder,window_size,overlap)
            #print('image',name,'cropped (size =',s_ima,')')
        else:
            commande='cp %s/%s %s'%(input_folder,name,output_folder)
            #print('image',name,'copied (size =',s_ima,')')
            os.system(commande)


In [55]:
crop_images(input_landslide_images,input_landslide_images_crop,256,0)

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/image_crop//zj073_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/image_crop//wn021_0005.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/image_crop//hz094_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/image_crop//hz094_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/imag

In [56]:
crop_images(input_landslide_dem_resized,input_landslide_dem_resized_crop,256,0)

creation of  ./Bijie-landslide-dataset/landslide/dem_resized_crop/


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//zj067_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//hz094_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//zj106_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//zj106_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslid

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//qxg055_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//ny027_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//ny026_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//qx005_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landsli

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//zj092_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//ny049_0004.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//ny049_0005.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/dem_resized_crop//zj093_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslid

In [57]:
crop_images(input_landslide_mask,input_landslide_mask_crop,256,0)

creation of  ./Bijie-landslide-dataset/landslide/mask_crop/


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//zj107_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//zj107_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//zj107_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qx101_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_cro

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//zj071_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qx103_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qx103_0005.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//zj111_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_cro

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg021_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg021_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg021_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg009_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg078_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg078_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg078_0004.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg078_0005.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//ny024_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//ny024_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//ny019_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//ny019_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_cro

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//qxg104_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//js057_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//hz006_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//zj023_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_cr

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//wn029_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//wn029_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//wn029_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//df008_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_cro

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//ny049_0004.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//ny049_0005.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//df022_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//wn003_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_cro

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//wn004_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//ny072_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//wn005_0001.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_crop//wn005_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/landslide/mask_cro

In [59]:
crop_images(input_nolandslide_images,input_nolandslide_images_crop,256,0)

creation of  ./Bijie-landslide-dataset/non-landslide/dem_resized_crop
creation of  ./Bijie-landslide-dataset/non-landslide/image_crop/


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/image_crop//fyb0_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/image_crop//fyb483_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/image_crop//fyb278_0005.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/image_crop//fyb278_0008.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-data

In [60]:
crop_images(input_nolandslide_dem_resized,input_nolandslide_dem_resized_crop,256,0)


/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb910_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb290_0008.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb253_0006.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb1203_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning:

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb719_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb296_0005.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb241_0008.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb255_0007.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: 

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb294_0008.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb1011_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb1005_0002.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb1005_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarnin

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb435_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dzjwv05274_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dnypl05014_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dzjwv05249_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: U

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb775_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb198_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dzjwv05242_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dnypl05023_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserW

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb994_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb599_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb1668_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dzjwv05284_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarn

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb1209_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dqxgpl05112_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb311_0008.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/dnypl05043_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: Use

/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb705_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb713_0003.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb1582_0008.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: ./Bijie-landslide-dataset/non-landslide/dem_resized_crop/fyb288_0008.png is a low contrast image
  io.imsave(name_save,window[2])
/var/folders/_v/vzmmycq54tx6qn34rbm0k8080000gn/T/ipykernel_10333/2438345671.py:19: UserWarning: